# Capstone Project 2: Online Retail Customer Purchasing Patterns & Segmentation

## 1. Problem Statement & Objective
- **Problem**: Understanding heterogeneous customer purchase behaviors, lifetime patterns, and purchase intervals to drive targeted retention and merchandising.
- **Objective**: Uncover empirical insights into customer transactions and perform unsupervised customer segmentation using Recency, Frequency, and Monetary (RFM) modeling.
- **Data Source**: ../../data_raw/OnlineRetail.csv (~541k transactions, 8 features: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country).


## 2. Environment Setup & Library Imports

In [ ]:
# Step 1: Import libraries for data wrangling, clustering, and dimensional reduction
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Clustering, preprocessing, and evaluation modules
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import joblib

# Plot styling configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Define workspace directories
DATA_RAW = Path('../../data_raw/OnlineRetail.csv')
DATA_PROCESSED_DIR = Path('../data_processed')
PLOTS_DIR = Path('../plots')
MODELS_DIR = Path('../models')

# Confirm path existence
DATA_RAW.exists()

## 3. Data Ingestion & Audit

In [ ]:
# Step 2: Load Online Retail dataset with appropriate character encoding
df_retail = pd.read_csv(DATA_RAW, encoding='ISO-8859-1')
df_retail['InvoiceDate'] = pd.to_datetime(df_retail['InvoiceDate'])

# Raw expression for Data Wrangler inspection (no print())
df_retail.head()

### Interpretation: Ingestion Audit
- The raw dataset registers ~541,909 transactional line-items.
- Missing values exist in CustomerID (~135k missing) and Description (~1.4k missing).
- We will inspect cancellations (Invoice numbers prefixed with 'C') and negative quantities in subsequent cleaning steps.

In [ ]:
# Step 3: Inspect raw summary statistics across numeric columns
df_retail.describe().T

### Interpretation: Negative Values & Data Integrity
- Negative values exist in Quantity (down to -80,995) and UnitPrice (down to -11,062).
- These represent returns, stock adjustments, and cancelled transactions, which require structured isolation prior to customer RFM aggregation.